# Etapa 2 — Pré-processamento sem cópias

Esta etapa descompacta os datasets em `Extraidos/`, remove diretórios descartáveis e cria `output/referencias_processamento.json`.

O JSON contém somente referências aos arquivos físicos que passaram pelos filtros de tipo e curso. Ele não replica os assessments nem os arquivos dos usuários. Os caminhos são relativos a `source_root`, o que mantém o índice portátil quando o projeto é movido.


In [ ]:
from pathlib import Path

# O notebook deve ser executado com o diretório de trabalho em Etapa_2.
# Estes são os únicos caminhos globais usados pelo pré-processamento.
RAIZ_PROJETO = Path.cwd().parent
PASTA_DATASETS = RAIZ_PROJETO / 'DataSets'
PASTA_EXTRAIDOS = RAIZ_PROJETO / 'Extraidos'
PASTA_SAIDA = Path.cwd() / 'output'
CAMINHO_REFERENCIAS = PASTA_SAIDA / 'referencias_processamento.json'
CAMINHO_CONTROLE_DESCOMPACTACAO = PASTA_SAIDA / 'controle_descompactacao.json'

# Filtros aplicados durante a construção do índice.
# Valores aceitos: 'exam', 'homework' ou 'todos'.
TIPO_ASSESSMENT_DESEJADO = 'exam'
NOME_CURSO_DESEJADO = 'todos'

print(f'Datasets: {PASTA_DATASETS.resolve()}')
print(f'Extraidos: {PASTA_EXTRAIDOS.resolve()}')
print(f'Referências: {CAMINHO_REFERENCIAS.resolve()}')
print(f'Controle de descompactação: {CAMINHO_CONTROLE_DESCOMPACTACAO.resolve()}')


## 1. Listar e descompactar os datasets

O arquivo `controle_descompactacao.json` é a fonte central da situação de cada dataset. O notebook lista primeiro o total encontrado, depois todos os pacotes já concluídos e, em seguida, todos os pacotes pendentes. Somente os pendentes são processados. O progresso da extração usa o formato `1/X pendentes`, enquanto a lista de concluídos usa `1/Y concluídos`.


In [ ]:
import json
import shutil
import tarfile
from datetime import datetime, timezone


def formatar_tamanho(numero_bytes: int) -> str:
    tamanho = float(numero_bytes)
    for unidade in ('B', 'KB', 'MB', 'GB', 'TB'):
        if tamanho < 1024 or unidade == 'TB':
            return f'{tamanho:.2f} {unidade}'
        tamanho /= 1024
    return f'{numero_bytes} B'


def medir_arquivos_regulares(pasta: Path) -> tuple[int, int]:
    """Retorna quantidade e tamanho dos arquivos presentes na pasta."""
    quantidade = 0
    tamanho_total = 0
    for caminho in pasta.rglob('*'):
        if caminho.is_file():
            quantidade += 1
            tamanho_total += caminho.stat().st_size
    return quantidade, tamanho_total


def obter_metadados_tar(arquivo: Path) -> tuple[int, int]:
    """Calcula quantidade e tamanho esperados dos arquivos do TAR."""
    with tarfile.open(arquivo, 'r:gz') as arquivo_tar:
        membros = arquivo_tar.getmembers()
    arquivos_regulares = [membro for membro in membros if membro.isfile()]
    return len(arquivos_regulares), sum(membro.size for membro in arquivos_regulares)


def carregar_controle() -> dict:
    """Carrega o único controle central de descompactação."""
    if not CAMINHO_CONTROLE_DESCOMPACTACAO.is_file():
        raise FileNotFoundError(
            f'Controle central não encontrado: {CAMINHO_CONTROLE_DESCOMPACTACAO}'
        )
    return json.loads(CAMINHO_CONTROLE_DESCOMPACTACAO.read_text(encoding='utf-8'))


def salvar_controle(controle: dict) -> None:
    """Atualiza o JSON central de forma atômica."""
    temporario = CAMINHO_CONTROLE_DESCOMPACTACAO.with_suffix('.json.tmp')
    temporario.write_text(
        json.dumps(controle, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    temporario.replace(CAMINHO_CONTROLE_DESCOMPACTACAO)


def registro_do_dataset(controle: dict, nome_dataset: str) -> dict:
    for registro in controle.get('datasets', []):
        if registro.get('nome') == nome_dataset:
            return registro
    raise KeyError(
        f'Dataset não cadastrado no controle central: {nome_dataset}'
    )


def extracao_esta_valida(
    registro: dict,
    arquivo: Path,
    pasta_destino: Path,
    quantidade_esperada: int,
    tamanho_esperado: int,
) -> bool:
    """Valida status central, origem e quantidade/tamanho extraídos."""
    if registro.get('status') != 'concluido' or not pasta_destino.is_dir():
        return False
    # Registros concluídos do controle inicial podem não conter os
    # metadados de tamanho. Nesse caso, preservamos o processamento anterior
    # e não reextraímos uma pasta existente só por falta desses campos.
    metadados_disponiveis = all(
        campo in registro
        for campo in (
            'arquivo_origem',
            'tamanho_comprimido_bytes',
            'quantidade_arquivos',
            'tamanho_descompactado_bytes',
        )
    )
    if not metadados_disponiveis:
        return pasta_destino.is_dir()
    if registro.get('arquivo_origem') != arquivo.name:
        return False
    if registro.get('tamanho_comprimido_bytes') != arquivo.stat().st_size:
        return False
    if registro.get('quantidade_arquivos') != quantidade_esperada:
        return False
    if registro.get('tamanho_descompactado_bytes') != tamanho_esperado:
        return False
    quantidade_atual, tamanho_atual = medir_arquivos_regulares(pasta_destino)
    return quantidade_atual == quantidade_esperada and tamanho_atual == tamanho_esperado


def atualizar_registro_concluido(
    registro: dict,
    arquivo: Path,
    quantidade_arquivos: int,
    tamanho_descompactado: int,
) -> None:
    registro.update({
        'status': 'concluido',
        'marcador_deve_existir': False,
        'arquivo_origem': arquivo.name,
        'tamanho_comprimido_bytes': arquivo.stat().st_size,
        'quantidade_arquivos': quantidade_arquivos,
        'tamanho_descompactado_bytes': tamanho_descompactado,
        'concluido_em': datetime.now(timezone.utc).isoformat(),
    })


def descompactar_tar_gz(
    arquivo: Path,
    pasta_destino: Path,
    registro: dict,
    controle: dict,
) -> bool:
    """Ignora somente quando o JSON central comprova uma extração completa."""
    quantidade_esperada, tamanho_esperado = obter_metadados_tar(arquivo)

    if extracao_esta_valida(
        registro,
        arquivo,
        pasta_destino,
        quantidade_esperada,
        tamanho_esperado,
    ):
        print(
            f'Validação central OK: {arquivo.name} | '
            f'{quantidade_esperada:,} arquivos | {formatar_tamanho(tamanho_esperado)}'
        )
        return False

    if pasta_destino.exists():
        print(f'Extração ausente ou incompleta; refazendo: {pasta_destino}')
        shutil.rmtree(pasta_destino)
    pasta_destino.mkdir(parents=True, exist_ok=True)

    try:
        with tarfile.open(arquivo, 'r:gz') as arquivo_tar:
            arquivo_tar.extractall(path=pasta_destino)

        quantidade_atual, tamanho_atual = medir_arquivos_regulares(pasta_destino)
        if quantidade_atual != quantidade_esperada or tamanho_atual != tamanho_esperado:
            raise RuntimeError(
                f'Validação pós-extração falhou para {arquivo.name}: '
                f'esperado {quantidade_esperada} arquivos/{tamanho_esperado} bytes, '
                f'encontrado {quantidade_atual} arquivos/{tamanho_atual} bytes.'
            )

        atualizar_registro_concluido(
            registro,
            arquivo,
            quantidade_esperada,
            tamanho_esperado,
        )
        salvar_controle(controle)
        print(
            f'Extração validada e controle atualizado: {arquivo.name} | '
            f'{quantidade_atual:,} arquivos | {formatar_tamanho(tamanho_atual)}'
        )
        return True
    except Exception:
        shutil.rmtree(pasta_destino, ignore_errors=True)
        raise


PASTA_EXTRAIDOS.mkdir(parents=True, exist_ok=True)
CONTROLE_DESCOMPACTACAO = carregar_controle()
ARQUIVOS_TAR_GZ = sorted(PASTA_DATASETS.rglob('*.tar.gz'))
ARQUIVOS_POR_NOME = {arquivo.name.removesuffix('.tar.gz'): arquivo for arquivo in ARQUIVOS_TAR_GZ}

# O JSON central é a fonte de verdade para decidir o que ainda falta.
registros = CONTROLE_DESCOMPACTACAO.get('datasets', [])
nomes_encontrados = set(ARQUIVOS_POR_NOME)
registros_concluidos = [registro for registro in registros if registro.get('status') == 'concluido']
registros_pendentes = [registro for registro in registros if registro.get('status') != 'concluido']
registros_sem_arquivo = [registro for registro in registros if registro.get('nome') not in nomes_encontrados]

print(f'Pacotes encontrados no DataSets: {len(ARQUIVOS_TAR_GZ):,}')
print(f'Pacotes concluídos segundo o controle central: {len(registros_concluidos):,}')
print(f'Pacotes pendentes segundo o controle central: {len(registros_pendentes):,}')

print('\nPacotes já concluídos:')
if registros_concluidos:
    for indice_concluido, registro in enumerate(registros_concluidos, start=1):
        print(f' - [{indice_concluido}/{len(registros_concluidos)} concluídos] {registro.get("nome")}')
else:
    print(' - Nenhum pacote concluído.')

if registros_sem_arquivo:
    print('Atenção: há registros no JSON sem arquivo .tar.gz correspondente:')
    for registro in registros_sem_arquivo:
        print(f" - {registro.get('nome')}")

if registros_pendentes:
    print('Pacotes que ainda faltam:')
    for registro in registros_pendentes:
        nome = registro.get('nome')
        print(f" - {nome} ({registro.get('status', 'sem_status')})")
else:
    print('Nenhum pacote pendente no controle central.')

pacotes_processados_agora = 0
for indice_pendente, registro in enumerate(registros_pendentes, start=1):
    nome_dataset = registro.get('nome')
    arquivo = ARQUIVOS_POR_NOME.get(nome_dataset)
    if arquivo is None:
        print(f'[{indice_pendente}/{len(registros_pendentes)} pendentes] Ausente no DataSets: {nome_dataset}')
        continue

    pasta_destino = PASTA_EXTRAIDOS / nome_dataset
    print(f'[{indice_pendente}/{len(registros_pendentes)} pendentes] Processando: {nome_dataset}')
    foi_descompactado = descompactar_tar_gz(
        arquivo,
        pasta_destino,
        registro,
        CONTROLE_DESCOMPACTACAO,
    )
    pacotes_processados_agora += int(foi_descompactado)
    situacao = 'Descompactado' if foi_descompactado else 'Já validado'
    print(
        f'[{indice_pendente}/{len(registros_pendentes)} pendentes] '
        f'{situacao}: {arquivo.name}'
    )

print()
print('Processamento concluído.')
print(f'Pacotes totais encontrados: {len(ARQUIVOS_TAR_GZ):,}')
print(f'Pacotes já concluídos antes desta execução: {len(registros_concluidos):,}')
print(f'Pacotes pendentes processados nesta execução: {pacotes_processados_agora:,}')
print(f'Pacotes restantes após esta execução: {sum(registro.get("status") != "concluido" for registro in CONTROLE_DESCOMPACTACAO.get("datasets", [])):,}')


## 2. Remover dados descartáveis

## 3. Criar referências dos arquivos filtrados

Os assessments são filtrados por tipo e curso. O padrão é `TIPO_ASSESSMENT_DESEJADO = 'exam'`; use `'homework'` ou `'todos'` para mudar o recorte. Cada arquivo de usuário é mantido somente quando seu par assessment–questão pertence a um assessment selecionado. Arquivos `grades` são filtrados pelo ID do assessment, pois uma grade como `grades/49.log` contém a nota agregada do assessment `49`.


In [ ]:
import json
import os
import re
import unicodedata
from collections import defaultdict


def normalizar(texto: str) -> str:
    sem_acento = ''.join(
        caractere
        for caractere in unicodedata.normalize('NFD', str(texto))
        if unicodedata.category(caractere) != 'Mn'
    )
    return sem_acento.strip().lower()


def ler_assessment_file(filepath: Path) -> dict[str, object]:
    dados: dict[str, object] = {'question_ids': []}
    em_exercicios = False
    try:
        for raw_line in filepath.read_text(encoding='utf-8').splitlines():
            line = raw_line.strip()
            if line.startswith('-- EXERCISES:'):
                em_exercicios = True
                continue
            if line.startswith('-- ASSESSMENT DATA:'):
                em_exercicios = False
                continue
            if em_exercicios:
                match = re.match(r'---- exercise \d+: (.+)', line)
                if match:
                    for question_id in match.group(1).split(' or '):
                        question_id = question_id.strip()
                        if question_id:
                            dados['question_ids'].append(question_id)
                continue
            match = re.match(r'---- (.+?): (.*)', line)
            if match:
                key, value = match.groups()
                if key in {'assessment title', 'class name', 'type'}:
                    dados[key.replace(' ', '_')] = value.strip()
    except (OSError, UnicodeError):
        return dados

    dados['question_ids'] = sorted(set(dados['question_ids']), key=lambda value: (not value.isdigit(), value))
    dados['path'] = filepath
    dados['filename'] = filepath.name
    return dados


def question_id_from_filename(filepath: Path) -> str | None:
    """Extrai o ID de questão dos nomes de executions, grades e codes.

    Execuções normalmente usam nomes como ``407_1326.log``; grades e
    códigos processados pela Etapa 1 usam frequentemente ``1326.log``.
    Ambos representam a questão 1326 e precisam ser indexados.
    """
    stem = filepath.stem
    if stem.isdigit():
        return stem
    if '_' not in stem:
        return None
    candidate = stem.rsplit('_', 1)[-1]
    return candidate if candidate.isdigit() else None


def reference(filepath: Path, root: Path, question_id: str | None = None) -> dict[str, str]:
    item = {
        'path': filepath.relative_to(root).as_posix(),
        'name': filepath.name,
    }
    if question_id is not None:
        item['question'] = question_id
    return item


def escrever_relatorio_assessments_selecionados(assessments: list[dict], root: Path) -> Path:
    """Salva os assessments usados no recorte atual para auditoria/filtros."""
    relatorio = {
        'descricao': (
            'Assessments selecionados pela Etapa 2. O relatório permite revisar '
            'títulos como Matrizes e Trabalho Prático Substitutivo antes de aplicar '
            'um filtro adicional.'
        ),
        'filtros_aplicados': {
            'tipo': TIPO_ASSESSMENT_DESEJADO,
            'curso': NOME_CURSO_DESEJADO,
        },
        'quantidade_assessments': len(assessments),
        'group_definitions': GRUPOS_ASSESSMENT,
        'assessments': [
            {
                **assessment,
                'caminho_absoluto': str((root / assessment['path']).resolve()),
                'titulo_normalizado': normalizar(assessment.get('assessment_title', '')),
                'group_description': assessment.get('group_description'),
            }
            for assessment in sorted(assessments, key=lambda item: item.get('path', ''))
        ],
    }
    caminho = PASTA_SAIDA / 'assessments_selecionados_detalhado.json'
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return caminho


GRUPOS_ASSESSMENT = {
    'TP 01': 'Variáveis e Estrutura Sequencial',
    'TP 02': 'Estrutura Condicional Simples e Composta',
    'TP 03': 'Estrutura Condicional Encadeada',
    'TP 04': 'Estrutura de Repetição por Condição',
    'TP 05': 'Vetores e Strings',
    'TP 06': 'Estrutura de Repetição por Contagem',
}

def classificar_grupo_assessment(titulo: str) -> str | None:
    """Classifica assessments nos grupos TP 01--06.

    A classificação usa o maior número TP encontrado no título. Assim,
    ``TP3 e TP4`` pertence ao grupo TP 04. Títulos sem grupo ou com termos
    explicitamente excluídos não entram no JSON.
    """
    texto = normalizar(titulo)
    texto_sem_acento = ''.join(
        caractere for caractere in unicodedata.normalize('NFD', texto)
        if unicodedata.category(caractere) != 'Mn'
    ).casefold()
    if not texto_sem_acento or texto_sem_acento == 'xxx':
        return None
    palavras_excluidas = (
        'matriz',
        'substitut',
        'prova final',
        'revisao',
    )
    if any(palavra in texto_sem_acento for palavra in palavras_excluidas):
        return None

    # Aceita TP, Trabalho Prático/Pratico e Avaliação/ Avaliacao numerados.
    padroes = (
        r'\btp\s*0*(\d+)\b',
        r'\btrabalho\s+pratic\w*\s*0*(\d+)\b',
        r'\bavaliac\w*\s+(?:parcial\s+)?0*(\d+)\b',
    )
    numeros = []
    for padrao in padroes:
        numeros.extend(int(valor) for valor in re.findall(padrao, texto_sem_acento))
    numeros_validos = [numero for numero in numeros if 1 <= numero <= 6]
    if not numeros_validos:
        return None
    numero = max(numeros_validos)
    return f'TP {numero:02d}'

def build_references(extracted_root: Path, output_path: Path, tipo: str, nome_curso_filtro: str) -> dict:
    """Constrói o índice em O(E + F log F), com uma única varredura do disco.

    E é o número de entradas visitadas em Extraidos e F é o número de referências
    mantidas no JSON. O filtro de assessments é resolvido antes de incluir arquivos
    de usuário, sem copiar nenhum conteúdo.
    """
    root = extracted_root.resolve()
    assessments = []
    selected_questions: set[str] = set()
    selected_assessment_questions: set[tuple[str, str, str]] = set()
    selected_scope_questions: set[tuple[str, str]] = set()
    # O ID numérico do assessment pode se repetir entre datasets/turmas;
    # grades usam escopo + assessment. Questões continuam globais por ID.
    # não pelo ID de uma questão.
    selected_assessment_ids: set[tuple[str, str]] = set()
    tipo_norm = normalizar(tipo)
    nome_curso_norm = normalizar(nome_curso_filtro)
    user_files: list[tuple[str, str, Path, str | None, str | None, str]] = []
    users_roots: dict[str, set[str]] = {}

    # Uma única caminhada coleta assessments e metadados dos usuários.
    for current_root, dirnames, filenames in os.walk(root):
        # Preserva as pastas mousemove no disco, mas não as percorre
        # nem inclui seu conteúdo no índice JSON.
        dirnames[:] = [nome for nome in dirnames if nome != 'mousemove']
        dirnames.sort()
        current = Path(current_root)
        relative_parts = current.relative_to(root).parts

        if current.name == 'assessments':
            for filename in sorted(filenames):
                if not filename.endswith('.data'):
                    continue
                filepath = current / filename
                dados = ler_assessment_file(filepath)
                assessment_type = str(dados.get('type', ''))
                class_name = str(dados.get('class_name', ''))
                if tipo_norm and tipo_norm != 'todos' and normalizar(assessment_type) != tipo_norm:
                    continue
                if nome_curso_norm != 'todos' and normalizar(class_name) != nome_curso_norm:
                    continue
                grupo_assessment = classificar_grupo_assessment(
                    str(dados.get('assessment_title', ''))
                )
                if grupo_assessment is None:
                    continue
                question_ids = sorted(
                    set(str(value) for value in dados['question_ids']),
                    key=lambda value: (not value.isdigit(), value),
                )
                selected_questions.update(question_ids)
                scope = filepath.parent.parent.relative_to(root).as_posix()
                selected_assessment_questions.update(
                    (scope, filepath.stem, question_id) for question_id in question_ids
                )
                selected_scope_questions.update(
                    (scope, question_id) for question_id in question_ids
                )
                # O arquivo 49.data corresponde a grades/49.log dentro do mesmo escopo.
                selected_assessment_ids.add((scope, filepath.stem))
                assessments.append({
                    'path': filepath.relative_to(root).as_posix(),
                    'filename': filepath.name,
                    'assessment_title': str(dados.get('assessment_title', '')),
                    'class_name': class_name,
                    'type': assessment_type,
                    'group': grupo_assessment,
                    'group_description': GRUPOS_ASSESSMENT[grupo_assessment],
                    'question_ids': question_ids,
                })

        # A entrada users é identificada pelo caminho relativo, sem iniciar
        # uma segunda caminhada para cada usuário.
        if 'users' in relative_parts:
            users_index = relative_parts.index('users')
            if users_index + 1 < len(relative_parts):
                user_id = relative_parts[users_index + 1]
                user_dir = root.joinpath(*relative_parts[:users_index + 2])
                users_roots.setdefault(user_id, set()).add(user_dir.relative_to(root).as_posix())
                for filename in sorted(filenames):
                    filepath = current / filename
                    relative_to_user = filepath.relative_to(user_dir).parts
                    kind = relative_to_user[0] if relative_to_user else ''
                    if kind not in {'codes', 'executions', 'grades', 'codemirror'}:
                        if filename == 'user.data':
                            kind = 'user_data'
                        elif filename.startswith('final_grade'):
                            kind = 'final_grades'
                        elif filename == 'logins.log':
                            kind = 'logins'
                        else:
                            continue
                    question_id = question_id_from_filename(filepath)
                    relative_parts_user = filepath.relative_to(root).parts
                    users_index_user = relative_parts_user.index('users')
                    scope = '/'.join(relative_parts_user[:users_index_user])
                    assessment_id = (
                        filepath.stem.split('_', 1)[0]
                        if '_' in filepath.stem and filepath.stem.split('_', 1)[0].isdigit()
                        else None
                    )
                    user_files.append((user_id, kind, filepath, question_id, assessment_id, scope))

    kinds = ('user_data', 'final_grades', 'codes', 'executions', 'grades', 'logins', 'codemirror')
    files_by_user: dict[str, dict[str, dict[str, dict[str, str]]]] = {}
    for user_id, kind, filepath, question_id, assessment_id, scope in user_files:
        # grades/49.log representa o assessment 49. Os demais arquivos
        # normalmente usam assessment_question (ex.: 407_1326.log); o filtro
        # precisa preservar somente pares pertencentes ao tipo selecionado.
        if kind == 'grades':
            if (scope, filepath.stem) not in selected_assessment_ids:
                continue
        elif kind in {'codes', 'executions', 'codemirror'}:
            # O arquivo usa assessment_question (ex.: 798_1326.log ou
            # 798_996.py). A ocorrência só entra se o assessment pai estiver
            # explicitamente entre os assessments selecionados (exam por padrão).
            if question_id is None or assessment_id is None:
                continue
            if (scope, assessment_id) not in selected_assessment_ids:
                continue
            if (scope, assessment_id, question_id) not in selected_assessment_questions:
                continue
        # Em grades, o número do arquivo é o assessment; não o ID da questão.
        item = reference(
            filepath,
            root,
            question_id if kind != 'grades' else None,
        )
        if kind == 'grades':
            item['assessment'] = filepath.stem
        files_by_user.setdefault(user_id, {}).setdefault(kind, {})[item['path']] = item

    users = []
    for user_id in sorted(files_by_user):
        filtered_files = {
            kind: sorted(items.values(), key=lambda item: item['path'])
            for kind, items in files_by_user[user_id].items()
            if items
        }
        # Um usuário só é útil para o processamento quando possui pelo menos
        # uma grade ou execution de um assessment selecionado. Arquivos de
        # código isolados, user.data ou logins não bastam para incluí-lo.
        arquivos_dados = (
            len(filtered_files.get('grades', []))
            + len(filtered_files.get('executions', []))
        )
        if arquivos_dados == 0:
            continue
        # Exibe somente raízes que contêm arquivos mantidos pelo filtro.
        roots_selecionadas = set()
        for itens in filtered_files.values():
            for item in itens:
                partes = Path(item['path']).parts
                if 'users' in partes:
                    indice_users = partes.index('users')
                    if indice_users + 1 < len(partes):
                        roots_selecionadas.add('/'.join(partes[:indice_users + 2]))
        users.append({
            'id': user_id,
            'roots': sorted(roots_selecionadas),
            'files': filtered_files,
        })

    source_root = os.path.relpath(root, output_path.resolve().parent).replace(os.sep, '/')
    statistics = {
        'assessments': len(assessments),
        'selected_questions': len(selected_questions),
        'users': len(users),
        'files_by_kind': {
            kind: sum(len(user.get('files', {}).get(kind, [])) for user in users)
            for kind in kinds
        },
    }
    data = {
        'schema_version': 1,
        'description': 'Referências relativas aos arquivos filtrados em Extraidos; não contém cópias dos dados.',
        'source_root': source_root,
        'filters': {'tipo': tipo, 'curso': nome_curso_filtro, 'groups': list(GRUPOS_ASSESSMENT)},
        'group_definitions': GRUPOS_ASSESSMENT,
        'assessments': sorted(assessments, key=lambda item: item['path']),
        'users': users,
        'statistics': statistics,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return data


referencias = build_references(PASTA_EXTRAIDOS, CAMINHO_REFERENCIAS, TIPO_ASSESSMENT_DESEJADO, NOME_CURSO_DESEJADO)
CAMINHO_ASSESSMENTS_DETALHADO = escrever_relatorio_assessments_selecionados(
    referencias['assessments'], Path(PASTA_EXTRAIDOS)
)
print(f'Referências criadas: {CAMINHO_REFERENCIAS.resolve()}')
print(f'Relatório de assessments selecionados: {CAMINHO_ASSESSMENTS_DETALHADO.resolve()}')
print(f"Assessments filtrados: {referencias['statistics']['assessments']}")
print(f"Questões selecionadas: {referencias['statistics']['selected_questions']}")
print(f"Usuários incluídos: {referencias['statistics']['users']}")
print(f"Arquivos referenciados: {sum(referencias['statistics']['files_by_kind'].values())}")


## 4. Validar o índice sem criar cópias

In [ ]:
import json

referencias = json.loads(CAMINHO_REFERENCIAS.read_text(encoding='utf-8'))
assert referencias['schema_version'] == 1
assert 'source_root' in referencias
assert all('path' in item for item in referencias.get('assessments', []))
assert all('id' in user and 'files' in user for user in referencias.get('users', []))
print('Validação concluída: JSON estruturado e sem arquivos copiados pela Etapa 2.')
